# Model_KD_Combined

**Knowledge Distillation — 2 Teachers × 2 Students × 2 Init Strategies = 8 Runs**

| Role | Model | Checkpoint |
|---|---|---|
| Teacher A | `VGG_Pretrained` | `vgg_pretrained_seed_85.pth` |
| Teacher B | `ResNet_Pretrained` | `resnet_pretrained_seed_52.pth` |
| Student 1 | `VWW_MobileNetV2` | `mobilenetv2_seed_74.pth` |
| Student 2 | `VWW_MobileNetV3` | `mobilenetv3_seed_74.pth` |

### Re-run behaviour
Every run checks its output checkpoint before training:
- **Checkpoint exists + accuracy ≥ threshold** → skipped automatically
- **Checkpoint exists + accuracy < threshold** → re-trains and overwrites
- **No checkpoint** → trains from scratch

Set `FORCE_RETRAIN = True` in the Config cell to ignore checkpoints and retrain everything.

**Prerequisites:** `Model_VGG_Pretrained`, `Model_ResNet_Pretrained`, `Model_MobileNetV2`, `Model_MobileNetV3`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place utils/ at: My Drive/stm32-thesis/utils/")

In [ ]:
import os, json
import torch

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import (
    VGG_Pretrained, ResNet_Pretrained,
    VWW_MobileNetV2, VWW_MobileNetV3,
)
from utils.train import setup_device, evaluate, train_kd

device = setup_device(seed=41)

In [ ]:
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")

---
## ⚙️ Config — edit here before each run

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────
SAVE_DIR = "/content/drive/My Drive/stm32-thesis/checkpoints"

# Teacher checkpoints — update seed suffix if needed
TEACHER_VGG_CKPT    = f"{SAVE_DIR}/vgg_pretrained_seed_85.pth"
TEACHER_RESNET_CKPT = f"{SAVE_DIR}/resnet_pretrained_seed_85.pth"  # update seed

# Student warm-start checkpoints
MV2_CKPT = f"{SAVE_DIR}/mobilenetv2_seed_74.pth"
MV3_CKPT = f"{SAVE_DIR}/mobilenetv3_seed_74.pth"

# ── KD hyperparameters ────────────────────────────────────────────────
KD_TEMPERATURE = 4.0
KD_ALPHA       = 0.7
KD_EPOCHS      = 60    # was 30 — scratch needs more room to converge
KD_LR_FT      = 3e-4  # fine-tune: lower to preserve warm-start weights
KD_LR_SCR     = 1e-3  # scratch: standard
KD_WD         = 1e-4
KD_PATIENCE   = 15    # was 10 — scratch val is noisy on 1500 samples

# ── Re-run control ────────────────────────────────────────────────────
# Minimum val accuracy to consider a checkpoint 'satisfactory'.
# If existing checkpoint is below this, it will be re-trained.
THRESHOLDS = {
    "vgg_mv2_ft"      : 0.800,
    "vgg_mv2_scratch" : 0.780,
    "vgg_mv3_ft"      : 0.800,
    "vgg_mv3_scratch" : 0.780,
    "resnet_mv2_ft"   : 0.800,
    "resnet_mv2_scratch": 0.780,
    "resnet_mv3_ft"   : 0.800,
    "resnet_mv3_scratch": 0.780,
}

# Set True to retrain everything regardless of existing checkpoints
FORCE_RETRAIN = False

---
## Helper — skip / run logic

In [ ]:
def should_run(run_key, ckpt_path, model_cls):
    """
    Returns (run: bool, reason: str, existing_acc: float | None)

    Skips the run if:
      - FORCE_RETRAIN is False, AND
      - checkpoint exists, AND
      - its val accuracy >= THRESHOLDS[run_key]
    """
    if FORCE_RETRAIN:
        return True, "FORCE_RETRAIN=True", None
    if not os.path.exists(ckpt_path):
        return True, "no checkpoint found", None
    # Load and evaluate existing checkpoint
    m = model_cls().to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device))
    acc = evaluate(m, val_loader, device)
    del m
    threshold = THRESHOLDS[run_key]
    if acc >= threshold:
        return False, f"acc {acc*100:.2f}% >= threshold {threshold*100:.1f}%", acc
    else:
        return True, f"acc {acc*100:.2f}% < threshold {threshold*100:.1f}% — retraining", acc


# Results registry — populated as runs complete or are skipped
results = {}

def register(run_key, acc, source="trained"):
    results[run_key] = {"val_acc": acc, "source": source}
    print(f"  → Registered [{run_key}]: {acc*100:.2f}%  ({source})")

---
## Load Teachers

In [ ]:
# ── Load and verify both teachers ───────────────────────────────────
teacher_vgg = VGG_Pretrained().to(device)
teacher_vgg.load_state_dict(torch.load(TEACHER_VGG_CKPT, map_location=device))
teacher_vgg.eval()
for p in teacher_vgg.parameters(): p.requires_grad = False
vgg_acc = evaluate(teacher_vgg, val_loader, device)
print(f"Teacher VGG_Pretrained    : {vgg_acc*100:.2f}%")

teacher_resnet = ResNet_Pretrained().to(device)
teacher_resnet.load_state_dict(torch.load(TEACHER_RESNET_CKPT, map_location=device))
teacher_resnet.eval()
for p in teacher_resnet.parameters(): p.requires_grad = False
resnet_acc = evaluate(teacher_resnet, val_loader, device)
print(f"Teacher ResNet_Pretrained : {resnet_acc*100:.2f}%")

# Baseline student accuracies (for delta reporting later)
_mv2_tmp = VWW_MobileNetV2().to(device)
_mv2_tmp.load_state_dict(torch.load(MV2_CKPT, map_location=device))
baseline_mv2 = evaluate(_mv2_tmp, val_loader, device); del _mv2_tmp

_mv3_tmp = VWW_MobileNetV3().to(device)
_mv3_tmp.load_state_dict(torch.load(MV3_CKPT, map_location=device))
baseline_mv3 = evaluate(_mv3_tmp, val_loader, device); del _mv3_tmp

print(f"\nStudent MobileNetV2 baseline: {baseline_mv2*100:.2f}%")
print(f"Student MobileNetV3 baseline: {baseline_mv3*100:.2f}%")

for teacher_name, t_acc, baseline in [
    ("VGG", vgg_acc, baseline_mv2), ("VGG", vgg_acc, baseline_mv3),
    ("ResNet", resnet_acc, baseline_mv2), ("ResNet", resnet_acc, baseline_mv3),
]:
    if t_acc <= baseline:
        print(f"⚠️  {teacher_name} teacher ({t_acc*100:.2f}%) is NOT stronger than student ({baseline*100:.2f}%) — KD may not help")

---
## Teacher: VGG_Pretrained  ·  Student: MobileNetV2


In [ ]:
# ── VGG_Pretrained → MobileNetV2 · KD Fine-Tune ────────────────────────
_ckpt = f"{SAVE_DIR}/mv2_kd_vgg_ft.pth"
_run, _reason, _existing_acc = should_run("vgg_mv2_ft", _ckpt, VWW_MobileNetV2)

if not _run:
    print(f"⏭️  Skipping vgg_mv2_ft — {_reason}")
    register("vgg_mv2_ft", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running vgg_mv2_ft — {_reason}")
    _student = VWW_MobileNetV2().to(device)
    _student.load_state_dict(torch.load(MV2_CKPT, map_location=device))
    print(f"MobileNetV2 warm-start baseline: {baseline_mv2*100:.2f}%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_vgg,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_FT,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("vgg_mv2_ft", _acc)

In [ ]:
# ── VGG_Pretrained → MobileNetV2 · KD Scratch ──────────────────────────
_ckpt = f"{SAVE_DIR}/mv2_kd_vgg_scratch.pth"
_run, _reason, _existing_acc = should_run("vgg_mv2_scratch", _ckpt, VWW_MobileNetV2)

if not _run:
    print(f"⏭️  Skipping vgg_mv2_scratch — {_reason}")
    register("vgg_mv2_scratch", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running vgg_mv2_scratch — {_reason}")
    _student = VWW_MobileNetV2().to(device)
    print(f"MobileNetV2 random init baseline: random weights, ~50%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_vgg,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_SCR,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("vgg_mv2_scratch", _acc)

---
## Teacher: VGG_Pretrained  ·  Student: MobileNetV3


In [ ]:
# ── VGG_Pretrained → MobileNetV3 · KD Fine-Tune ────────────────────────
_ckpt = f"{SAVE_DIR}/mv3_kd_vgg_ft.pth"
_run, _reason, _existing_acc = should_run("vgg_mv3_ft", _ckpt, VWW_MobileNetV3)

if not _run:
    print(f"⏭️  Skipping vgg_mv3_ft — {_reason}")
    register("vgg_mv3_ft", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running vgg_mv3_ft — {_reason}")
    _student = VWW_MobileNetV3().to(device)
    _student.load_state_dict(torch.load(MV3_CKPT, map_location=device))
    print(f"MobileNetV3 warm-start baseline: {baseline_mv3*100:.2f}%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_vgg,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_FT,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("vgg_mv3_ft", _acc)

In [ ]:
# ── VGG_Pretrained → MobileNetV3 · KD Scratch ──────────────────────────
_ckpt = f"{SAVE_DIR}/mv3_kd_vgg_scratch.pth"
_run, _reason, _existing_acc = should_run("vgg_mv3_scratch", _ckpt, VWW_MobileNetV3)

if not _run:
    print(f"⏭️  Skipping vgg_mv3_scratch — {_reason}")
    register("vgg_mv3_scratch", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running vgg_mv3_scratch — {_reason}")
    _student = VWW_MobileNetV3().to(device)
    print(f"MobileNetV3 random init baseline: random weights, ~50%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_vgg,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_SCR,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("vgg_mv3_scratch", _acc)

---
## Teacher: ResNet_Pretrained  ·  Student: MobileNetV2


In [ ]:
# ── ResNet_Pretrained → MobileNetV2 · KD Fine-Tune ────────────────────────
_ckpt = f"{SAVE_DIR}/mv2_kd_resnet_ft.pth"
_run, _reason, _existing_acc = should_run("resnet_mv2_ft", _ckpt, VWW_MobileNetV2)

if not _run:
    print(f"⏭️  Skipping resnet_mv2_ft — {_reason}")
    register("resnet_mv2_ft", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running resnet_mv2_ft — {_reason}")
    _student = VWW_MobileNetV2().to(device)
    _student.load_state_dict(torch.load(MV2_CKPT, map_location=device))
    print(f"MobileNetV2 warm-start baseline: {baseline_mv2*100:.2f}%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_resnet,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_FT,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("resnet_mv2_ft", _acc)

In [ ]:
# ── ResNet_Pretrained → MobileNetV2 · KD Scratch ──────────────────────────
_ckpt = f"{SAVE_DIR}/mv2_kd_resnet_scratch.pth"
_run, _reason, _existing_acc = should_run("resnet_mv2_scratch", _ckpt, VWW_MobileNetV2)

if not _run:
    print(f"⏭️  Skipping resnet_mv2_scratch — {_reason}")
    register("resnet_mv2_scratch", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running resnet_mv2_scratch — {_reason}")
    _student = VWW_MobileNetV2().to(device)
    print(f"MobileNetV2 random init baseline: random weights, ~50%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_resnet,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_SCR,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("resnet_mv2_scratch", _acc)

---
## Teacher: ResNet_Pretrained  ·  Student: MobileNetV3


In [ ]:
# ── ResNet_Pretrained → MobileNetV3 · KD Fine-Tune ────────────────────────
_ckpt = f"{SAVE_DIR}/mv3_kd_resnet_ft.pth"
_run, _reason, _existing_acc = should_run("resnet_mv3_ft", _ckpt, VWW_MobileNetV3)

if not _run:
    print(f"⏭️  Skipping resnet_mv3_ft — {_reason}")
    register("resnet_mv3_ft", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running resnet_mv3_ft — {_reason}")
    _student = VWW_MobileNetV3().to(device)
    _student.load_state_dict(torch.load(MV3_CKPT, map_location=device))
    print(f"MobileNetV3 warm-start baseline: {baseline_mv3*100:.2f}%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_resnet,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_FT,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("resnet_mv3_ft", _acc)

In [ ]:
# ── ResNet_Pretrained → MobileNetV3 · KD Scratch ──────────────────────────
_ckpt = f"{SAVE_DIR}/mv3_kd_resnet_scratch.pth"
_run, _reason, _existing_acc = should_run("resnet_mv3_scratch", _ckpt, VWW_MobileNetV3)

if not _run:
    print(f"⏭️  Skipping resnet_mv3_scratch — {_reason}")
    register("resnet_mv3_scratch", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running resnet_mv3_scratch — {_reason}")
    _student = VWW_MobileNetV3().to(device)
    print(f"MobileNetV3 random init baseline: random weights, ~50%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_resnet,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_SCR,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("resnet_mv3_scratch", _acc)

---
## Results Summary

In [ ]:
# ── Reload all best checkpoints and build final table ───────────────
def reload_acc(model_cls, ckpt_name):
    path = f"{SAVE_DIR}/{ckpt_name}"
    if not os.path.exists(path):
        return None
    m = model_cls().to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    return evaluate(m, val_loader, device)

final = {
    "vgg_mv2_ft"       : reload_acc(VWW_MobileNetV2, "mv2_kd_vgg_ft.pth"),
    "vgg_mv2_scratch"  : reload_acc(VWW_MobileNetV2, "mv2_kd_vgg_scratch.pth"),
    "vgg_mv3_ft"       : reload_acc(VWW_MobileNetV3, "mv3_kd_vgg_ft.pth"),
    "vgg_mv3_scratch"  : reload_acc(VWW_MobileNetV3, "mv3_kd_vgg_scratch.pth"),
    "resnet_mv2_ft"    : reload_acc(VWW_MobileNetV2, "mv2_kd_resnet_ft.pth"),
    "resnet_mv2_scratch": reload_acc(VWW_MobileNetV2, "mv2_kd_resnet_scratch.pth"),
    "resnet_mv3_ft"    : reload_acc(VWW_MobileNetV3, "mv3_kd_resnet_ft.pth"),
    "resnet_mv3_scratch": reload_acc(VWW_MobileNetV3, "mv3_kd_resnet_scratch.pth"),
}

def fmt(acc, baseline):
    if acc is None: return f"{'—':>9}   {'—':>7}"
    delta = acc - baseline
    sign  = '+' if delta >= 0 else ''
    return f"{acc*100:>8.2f}%  ({sign}{delta*100:.2f}%)"

W = 72
print("=" * W)
print(f"{'Run':<35} {'Val Acc':>9}  {'Δ baseline':>10}")
print("-" * W)
print(f"{'Teacher VGG_Pretrained':<35} {vgg_acc*100:>8.2f}%")
print(f"{'Teacher ResNet_Pretrained':<35} {resnet_acc*100:>8.2f}%")
print("-" * W)
print(f"{'MobileNetV2 baseline':<35} {baseline_mv2*100:>8.2f}%")
print(f"  VGG    → MobileNetV2  FT      {fmt(final['vgg_mv2_ft'],    baseline_mv2)}")
print(f"  VGG    → MobileNetV2  Scratch  {fmt(final['vgg_mv2_scratch'], baseline_mv2)}")
print(f"  ResNet → MobileNetV2  FT      {fmt(final['resnet_mv2_ft'],    baseline_mv2)}")
print(f"  ResNet → MobileNetV2  Scratch  {fmt(final['resnet_mv2_scratch'], baseline_mv2)}")
print("-" * W)
print(f"{'MobileNetV3 baseline':<35} {baseline_mv3*100:>8.2f}%")
print(f"  VGG    → MobileNetV3  FT      {fmt(final['vgg_mv3_ft'],    baseline_mv3)}")
print(f"  VGG    → MobileNetV3  Scratch  {fmt(final['vgg_mv3_scratch'], baseline_mv3)}")
print(f"  ResNet → MobileNetV3  FT      {fmt(final['resnet_mv3_ft'],    baseline_mv3)}")
print(f"  ResNet → MobileNetV3  Scratch  {fmt(final['resnet_mv3_scratch'], baseline_mv3)}")
print("=" * W)